In [ ]:
# Instalamos las librerías necesarias
# 'k3d' es útil para interactividad en notebooks, pero para este ejemplo
# usaremos generación de video que es más robusta.
!pip install trimesh vedo numpy matplotlib
!apt-get install ffmpeg # Necesario para guardar videos/gifs de alta calidad

In [ ]:
import trimesh
import vedo
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# 1. CONFIGURACIÓN INICIAL
# ---------------------------------------------------------
# Configurar vedo para renderizado off-screen (necesario en Colab para video)
vedo.settings.default_backend = 'vtk' 
vedo.settings.screenshot_transparent_background = False

def analizar_y_visualizar(ruta_modelo=None):
    print("--- 1. CARGANDO MODELO ---")
    
    # Si no se provee ruta, descargamos un modelo de prueba (un cubo o una tetera)
    if ruta_modelo is None:
        print("Descargando modelo de prueba (Stanford Bunny)...")
        # Usamos vedo para descargar un ejemplo, devuelve la ruta del archivo
        ruta_modelo = vedo.download("bunny.obj")
    
    # ---------------------------------------------------------
    # 2. ANÁLISIS ESTRUCTURAL CON TRIMESH
    # ---------------------------------------------------------
    # Usamos Trimesh porque es excelente para obtener datos puros
    mesh_data = trimesh.load(ruta_modelo)
    
    # Aseguramos que sea una malla única (a veces cargan como 'Scene')
    if isinstance(mesh_data, trimesh.Scene):
        # Si es una escena, tomamos todas las geometrías y las unimos
        geometries = list(mesh_data.geometry.values())
        mesh_data = trimesh.util.concatenate(geometries)

    print(f"\n--- INFORMACIÓN ESTRUCTURAL ---")
    print(f"Archivo: {ruta_modelo}")
    print(f"🔸 Vértices (Puntos): {mesh_data.vertices.shape[0]}")
    print(f"🔹 Caras (Polígonos): {mesh_data.faces.shape[0]}")
    print(f"📏 Aristas (Edges):    {mesh_data.edges.shape[0]}")
    print(f"Volumen cerrado:      {mesh_data.is_watertight}")
    print("-------------------------------\n")

    # ---------------------------------------------------------
    # 3. PREPARACIÓN VISUAL CON VEDO
    # ---------------------------------------------------------
    print("--- 2. GENERANDO VISUALIZACIÓN ---")
    
    # A. LAS CARAS (El objeto sólido)
    # Convertimos el objeto trimesh a vedo para renderizar
    vedo_mesh = vedo.Mesh(mesh_data)
    vedo_mesh.c("gray").alpha(0.5) # Color gris, semitransparente
    
    # B. LAS ARISTAS (Wireframe)
    # Extraemos el wireframe y lo pintamos de azul
    edges_vis = vedo_mesh.clone().wireframe(True).c("blue4").lw(1)
    
    # C. LOS VÉRTICES (Puntos)
    # Creamos una nube de puntos en las coordenadas de los vértices
    verts_vis = vedo.Points(vedo_mesh.vertices).c("red").ps(3) # ps = point size

    # Crear textos informativos 3D
    txt = vedo.Text2D(
        f"Vertices: {mesh_data.vertices.shape[0]}\nCaras: {mesh_data.faces.shape[0]}",
        pos="top-left", c="black", s=0.8
    )

    # ---------------------------------------------------------
    # 4. ANIMACIÓN Y EXPORTACIÓN (Bonus)
    # ---------------------------------------------------------
    print("Generando animación (GIF)... por favor espera.")
    
    # Inicializamos el Plotter (el lienzo 3D)
    # offscreen=True es vital en Colab para no crashear
    plt_3d = vedo.Plotter(offscreen=True, size=(600, 600), bg="white")
    
    # Añadimos los elementos: Caras, Aristas, Vértices y Texto
    plt_3d.show(vedo_mesh, edges_vis, verts_vis, txt, camera={'pos':(0, 0, 3)})

    # Configuración de video
    video = vedo.Video("animacion_modelo.gif", duration=4, backend='ffmpeg')

    # Bucle de animación (Rotación 360 grados)
    for i in range(40):
        # Rotamos todos los componentes
        vedo_mesh.rotate_y(9) 
        edges_vis.rotate_y(9)
        verts_vis.rotate_y(9)
        
        # Renderizamos el frame y lo añadimos al video
        plt_3d.render()
        video.add_frame()

    video.close() # Cierra y guarda el archivo
    print("✅ GIF guardado como 'animacion_modelo.gif'")

    # 5. MOSTRAR RESULTADO EN COLAB
    plt.figure(figsize=(8, 8))
    plt.axis('off')
    plt.title("Vista Previa del Modelo")
    plt.imshow(plt.imread("animacion_modelo.gif"))
    plt.show()

# --- EJECUTAR ---


# Por defecto, usamos el ejemplo:
analizar_y_visualizar(r'/content/eyeball.obj')